# RazorMind AI: Machine Learning Fraud Detection Engine
### Inspired by Razorpay Thirdwatch & Real-Time Risk Intelligence
**Objective**: Build, evaluate, and interpret an ensemble ML classifier to detect fraudulent payment transactions (stolen cards, velocity bursts, account takeover) while minimizing false positive declines for legitimate merchants.

## 1. Environment Setup & Data Loading

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve, precision_recall_curve, average_precision_score
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
import joblib

# Set aesthetic styling
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
np.random.seed(42)

# Load synthetic FinTech dataset
data_path = os.path.join("..", "data", "raw", "transactions.csv")
if not os.path.exists(data_path):
    print("Generating data first...")
    import sys
    sys.path.append(os.path.join("..", "data"))
    import generate_data
    m = generate_data.generate_merchants()
    c = generate_data.generate_customers()
    df = generate_data.generate_transactions(m, c)
else:
    df = pd.read_csv(data_path)

print(f"Dataset Shape: {df.shape}")
df.head()

## 2. Exploratory Data Analysis (EDA) & Imbalance Check

In [ ]:
fraud_counts = df['is_fraud'].value_counts()
print(f"Non-Fraud: {fraud_counts[0]} ({fraud_counts[0]/len(df)*100:.2f}%)")
print(f"Fraud: {fraud_counts[1]} ({fraud_counts[1]/len(df)*100:.2f}%)")

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Plot 1: Target distribution
sns.countplot(data=df, x='is_fraud', palette=['#10b981', '#ef4444'], ax=axes[0])
axes[0].set_title("Fraud Distribution (Class Imbalance)", fontsize=13, fontweight='bold')
axes[0].set_xticklabels(['Legitimate (0)', 'Fraud (1)'])

# Plot 2: Amount distribution by fraud
sns.boxplot(data=df, x='is_fraud', y='amount', palette=['#10b981', '#ef4444'], ax=axes[1])
axes[1].set_yscale('log')
axes[1].set_title("Transaction Amount (Log Scale)", fontsize=13, fontweight='bold')
axes[1].set_xticklabels(['Legitimate', 'Fraud'])

# Plot 3: Fraud rate across payment methods
method_fraud = df.groupby('payment_method')['is_fraud'].mean().reset_index()
sns.barplot(data=method_fraud, x='payment_method', y='is_fraud', palette='mako', ax=axes[2])
axes[2].set_title("Fraud Rate by Payment Method", fontsize=13, fontweight='bold')
axes[2].set_ylabel("Fraud Probability")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

## 3. Feature Engineering & Preparation

In [ ]:
NUMERIC_FEATURES = [
    "amount", "hour_of_day", "day_of_week", "velocity_1h", 
    "velocity_24h", "amount_to_avg_ratio", "geo_distance_km", "is_vpn_or_proxy"
]
CATEGORICAL_FEATURES = ["payment_method", "device_type"]

X = df[NUMERIC_FEATURES + CATEGORICAL_FEATURES]
y = df["is_fraud"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set: {X_train.shape}, Test set: {X_test.shape}")
print(f"Fraud cases in test set: {y_test.sum()}")

## 4. Model Training & Comparison
We compare a Baseline Logistic Regression against cost-sensitive Random Forest and Gradient Boosting.

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", "passthrough", NUMERIC_FEATURES),
        ("cat", OneHotEncoder(handle_unknown="ignore"), CATEGORICAL_FEATURES)
    ]
)

models = {
    "Logistic Regression (Balanced)": LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42),
    "Random Forest (Balanced)": RandomForestClassifier(n_estimators=120, max_depth=12, min_samples_split=4, class_weight="balanced", random_state=42, n_jobs=-1),
    "Gradient Boosting": GradientBoostingClassifier(n_estimators=120, learning_rate=0.08, max_depth=5, random_state=42)
}

results = {}
for name, clf in models.items():
    pipe = Pipeline([("prep", preprocessor), ("clf", clf)])
    pipe.fit(X_train, y_train)
    y_prob = pipe.predict_proba(X_test)[:, 1]
    y_pred = pipe.predict(X_test)
    auc = roc_auc_score(y_test, y_prob)
    pr_auc = average_precision_score(y_test, y_prob)
    results[name] = {"pipeline": pipe, "auc": auc, "pr_auc": pr_auc, "y_prob": y_prob, "y_pred": y_pred}
    print(f"[{name}] -> ROC-AUC: {auc:.4f} | PR-AUC: {pr_auc:.4f}")

## 5. ROC & Precision-Recall Curves Evaluation

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ROC Curves
for name, res in results.items():
    fpr, tpr, _ = roc_curve(y_test, res["y_prob"])
    axes[0].plot(fpr, tpr, label=f"{name} (AUC = {res['auc']:.3f})")
axes[0].plot([0, 1], [0, 1], 'k--', alpha=0.5)
axes[0].set_title("ROC Curves", fontsize=13, fontweight='bold')
axes[0].set_xlabel("False Positive Rate")
axes[0].set_ylabel("True Positive Rate")
axes[0].legend()

# Precision-Recall Curves
for name, res in results.items():
    prec, rec, _ = precision_recall_curve(y_test, res["y_prob"])
    axes[1].plot(rec, prec, label=f"{name} (PR-AUC = {res['pr_auc']:.3f})")
axes[1].set_title("Precision-Recall Curves (Critical for Rare Fraud)", fontsize=13, fontweight='bold')
axes[1].set_xlabel("Recall")
axes[1].set_ylabel("Precision")
axes[1].legend()

plt.tight_layout()
plt.show()

## 6. Confusion Matrix & Decision Threshold Optimization

In [ ]:
best_pipe = results["Random Forest (Balanced)"]["pipeline"]
y_probs = results["Random Forest (Balanced)"]["y_prob"]

# Optimize threshold: standard 0.5 vs custom risk cutoff 0.40
chosen_threshold = 0.40
y_custom_pred = (y_probs >= chosen_threshold).astype(int)

cm = confusion_matrix(y_test, y_custom_pred)

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Predicted Legitimate', 'Predicted Fraud'],
            yticklabels=['Actual Legitimate', 'Actual Fraud'])
plt.title(f"Confusion Matrix (Threshold = {chosen_threshold})", fontsize=13, fontweight='bold')
plt.show()

print(classification_report(y_test, y_custom_pred, target_names=['Legitimate', 'Fraud']))

## 7. Explainability & Feature Importance

In [ ]:
rf_model = best_pipe.named_steps["clf"]
ohe = best_pipe.named_steps["prep"].named_transformers_["cat"]
ohe_feature_names = list(ohe.get_feature_names_out(CATEGORICAL_FEATURES))
all_feature_names = NUMERIC_FEATURES + ohe_feature_names

importances = rf_model.feature_importances_
feat_df = pd.DataFrame({"feature": all_feature_names, "importance": importances}).sort_values("importance", ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(data=feat_df.head(10), x='importance', y='feature', palette='viridis')
plt.title("Top 10 Feature Importances (Razorpay Risk Factors)", fontsize=13, fontweight='bold')
plt.xlabel("Gini Importance")
plt.tight_layout()
plt.show()

feat_df.head(10)

## 8. Exporting Production Pipeline

In [ ]:
artifact_dir = os.path.join("..", "backend", "artifacts")
os.makedirs(artifact_dir, exist_ok=True)
export_path = os.path.join(artifact_dir, "fraud_model.joblib")

joblib.dump(best_pipe, export_path)
print(f"Production Fraud Pipeline exported to: {export_path}")